# تمرين Lecture 02: تعلّم صيغة الإجابة باستخدام LoRA ومقارنة النتائج قبل التدريب وبعده

سنستخدم أسئلة وأجوبة متكررة (FAQ) لدورة افتراضية لتدريب نموذج صغير مُعدّ لاتباع التعليمات باستخدام LoRA لمدة قصيرة، ثم نقارن **ما تغيّر وما لم يتغيّر** قبل التدريب وبعده. نفصل في التقييم بين **معدل الالتزام بالصيغة** (هل التزم بصيغة JSON التي تحتوي على `answer` و`needs_review`؟) و**دقة المحتوى**.

**ما الذي ستتحقق منه في هذا التمرين؟**

1. قد يعجز نموذج صغير عن اتباع تعليمات الصيغة حتى عندما ترد في الموجّه (القسم 3، قبل التدريب).
2. يصف التمرين الياباني الأصلي اتباع النموذج للصيغة على نحو شبه منتظم بعد دقائق قليلة من التدريب باستخدام LoRA؛ افحص النتيجة مع البيانات العربية (القسم 5، بعد التدريب).
3. الالتزام بالصيغة لا يضمن حكمًا سليمًا عند السؤال عن معلومات غير موجودة في FAQ (هل يطلب التأكيد أم يقرر إجابة؟ القسمان 5 و6).
4. للتوسّع: تحميل النموذج نفسه بعد تكميمه إلى 4 بتات يغيّر مخرجاته حتى دون تدريب (القسم 8).

**مسار العمل**

| القسم | المحتوى | الوقت التقريبي (T4) |
| --- | --- | --- |
| 0–3 | البيئة والإعدادات وتوليد البيانات وتحميل النموذج والتقييم قبل التدريب | نحو 5 دقائق |
| 4–7 | التدريب والتقييم بعده وأسئلة المعرفة وجدول المقارنة والتسليم | نحو 10 دقائق (منها 4–5 دقائق للتدريب) |
| 8 (اختياري) | التحميل بدقة 4 بتات والمقارنة | نحو دقيقتين |

يساعدك تشغيل الأقسام 0–3 قبل المحاضرة للتحقق من البيئة على تجنب التأخير أثناءها (اختياري). يفقد Colab المتغيرات عند انقطاع الجلسة؛ إذا حدث ذلك، فابدأ من القسم 0 مرة أخرى يوم المحاضرة.

**المتطلبات**: Google Colab ‏(وحدة GPU من نوع T4، ضمن الحصة المجانية). النموذج: `Qwen/Qwen3.5-0.8B` ‏(Apache-2.0). ينشئ هذا الدفتر بيانات افتراضية، ولا يتضمن مواعيد تسليم أو عناوين URL حقيقية.

**طريقة قراءة الدفتر**: يوضح كل قسم «ما ستفعله» و«ما ستبحث عنه» و«حل المشكلات». شغّل الخلايا بالترتيب من الأعلى. يؤدي تخطي الخلايا إلى أخطاء متغيرات غير معرّفة في الخلايا اللاحقة.

**ملاحظة بشأن الترجمة**: قياسات الوقت والخسارة والمخرجات المذكورة هنا مأخوذة من التمرين الياباني الأصلي. قد تختلف النتائج عند استخدام FAQ والأسئلة المترجمة؛ ولم تُعَد هذه القياسات لهذه الترجمة.

الإصدار: 1.1 ‏(2026-10-18)

## 0. التحقق من البيئة وتثبيت المكتبات

**ما ستفعله**: تأكد من تخصيص وحدة GPU، وثبّت إصدارات المكتبات التي جرى التحقق من عملها.

**ما ستبحث عنه**

- تعرض الخلية الأولى `Tesla T4`.
- تنتهي الخلية الثانية بقائمة الإصدارات، مثل `transformers: 5.18.0`. يستغرق التثبيت نحو دقيقة.

**حل المشكلات**

- إذا ظهرت الرسالة «لم يُعثر على GPU»، فاختر **Runtime → Change runtime type → T4 GPU** وأعد تشغيل الخلية. إذا منع الضغط على الخدمة تخصيص الوحدة، فحاول لاحقًا.
- تحذّر رسائل pip الحمراء التي تبدأ بـ `ERROR: pip's dependency resolver ...` (بشأن `cudf` أو `pyarrow`) من تعارضات مع حزم مثبّتة مسبقًا في Colab. لا تؤثر في هذا التمرين؛ تابع العمل.

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # لا تتوفر nvidia-smi في بيئات التشغيل التي تستخدم CPU.
    gpu_info = ""
print(gpu_info or "لم يُعثر على GPU. اختر Runtime → Change runtime type → T4 GPU.")

In [ ]:
# إصدارات جرى التحقق منها على T4 في Colab ‏(Python 3.13 وtorch 2.11.0+cu130). استخدم torch المرفقة مع Colab.
PINS = {
    "transformers": "5.18.0",   # يتطلب Qwen3.5 الإصدار 5.x من transformers.
    "peft": "0.21.1",           # تنفيذ LoRA.
    "trl": "1.14.2",            # حلقة التدريب (SFTTrainer).
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # الإصدار torchao ‏(0.10) المرفق غير متوافق مع peft 0.21 ويسبب ImportError أثناء التدريب. أزله لأنه غير مستخدم.
%pip install -q {spec}

# تظهر تحذيرات الاعتماديات بشأن cudf-cu13 وpyarrow بصيغة ERROR؛ يمكن تجاهلها لأن cudf غير مستخدمة.
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. الإعدادات

**ما ستفعله**: اجمع اسم النموذج وعدد الحقب التدريبية (مرات المرور على البيانات) وسائر الإعدادات في قاموس واحد اسمه `CONFIG`. سيُسجَّل هذا القاموس كما هو في ملف التسليم في القسم 7، لتعرف لاحقًا «الشروط التي أُجري القياس فيها».

- تثبيت `revision` يضمن استخدام الأوزان نفسها عند كل تشغيل، ليبقى شرط المقارنة ثابتًا قبل التدريب وبعده وبين المشاركين.
- تعني `num_train_epochs: 20` المرور على بيانات التدريب (14 مثالًا) عشرين مرة. ومع خطوتين في كل مرور، يصبح المجموع 40 خطوة استغرقت 4–5 دقائق على T4 في التشغيل الياباني الأصلي؛ قد يختلف الزمن مع البيانات العربية.
- يستخدم `gen: {"do_sample": False}` التوليد الجشع. بذلك تثبت ظروف التوليد قبل التدريب وبعده ويزول أثر تفاوت أخذ العينات.

إذا غيّرت `USE_DRIVE` إلى `True`، فسيُركَّب Google Drive ويُحفَظ فيه المهايئ المدرَّب وملفات النتائج أيضًا. استخدم ذلك للاحتفاظ بالنتائج إذا انقطعت الجلسة. اترك القيمة `False` إن كنت ستنهي العمل في جلسة واحدة.

**ما ستبحث عنه**: تُعرض الإعدادات بصيغة JSON.

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # نموذج صغير مُعدّ لاتباع التعليمات (Apache-2.0). يتسع على T4 دون تكميم.
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # ثُبّت لاستخدام الأوزان نفسها في كل تشغيل.
    "seed": 42,
    "num_train_epochs": 20,  # عدد مرات المرور على بيانات التدريب (14 مثالًا). خطوتان لكل مرور = 40 خطوة (استغرقت 4–5 دقائق على T4 في التشغيل الياباني الأصلي).
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # بُعد مصفوفة LoRA (القيمة الأصغر تعني معلمات أقل للتدريب).
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # تثبيت التقييم على التوليد الجشع (شروط متسقة قبل التدريب وبعده).
    "work_dir": "/content/qlora_lab",                          # مكان ملفات النتائج (تظهر في قائمة الملفات يسارًا).
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # مكان الحفظ عندما تكون USE_DRIVE=True.
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # اضبطها على True لتركيب Google Drive وحفظ المهايئات والمخرجات.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. إنشاء FAQ افتراضية وبيانات التدريب

**ما ستفعله**: أنشئ بيانات التدريب (`train`) والتحقق (`valid`) والتقييم (`test`) من FAQ افتراضية تتضمن خمسة بنود.

مدخل النموذج هو «FAQ + السؤال»، ومخرجه المطلوب سطر JSON واحد: `{"answer": <string>, "needs_review": <boolean>}`.

- إذا كان للسؤال دليل في FAQ، فأجب بنص البند المعني واجعل `needs_review: false`.
- إذا لم يكن له دليل، فأجب «لا تتضمن هذه الأسئلة المتكررة تلك المعلومة. يلزم التأكد من منظمي الدورة». واجعل `needs_review: true`.

في مثال اسم الملف `student_id_assignment_number.pdf`، استبدل `student_id` برقمك الجامعي و`assignment_number` برقم الواجب، مع الإبقاء على الشرطة السفلية والامتداد `.pdf`.

**تقسيم البيانات**

- تُوزَّع الصياغات المختلفة للسؤال عن بند FAQ نفسه على `train` و`valid` و`test` **بحسب القالب**، فلا يظهر نص التدريب نفسه حرفيًا في الاختبار.
- يتضمن `test` عدد 21 سؤالًا (15 لها دليل، و6 بلا دليل). تكمل أسئلة مخصصة للاختبار فقط هذا العدد.
- تُحفظ الإجابات المتوقعة (قيمة `needs_review` والكلمات المفتاحية المطلوبة) منفصلةً في `gold`، ولا تُمرَّر إلى النموذج.
- يتضمن `train` عدد 14 مثالًا (10 لها دليل، و4 بلا دليل)، و`valid` عدد 7 أمثلة. ومع وجود خمسة بنود FAQ فقط، تظل بيانات التدريب محدودة، ولذلك يتدرب النموذج على الأمثلة نفسها مرات متكررة. يُضبط عدد مرات التكرار بعدد الحقب (`num_train_epochs`) في القسم 4، دون تكرار البيانات نفسها. المنهج المعتاد هو وضع الأمثلة الفعلية فقط في ملفات البيانات وضبط حجم التدريب من إعداداته.

**ما ستبحث عنه**: تظهر `train=14 valid=7 test=21 gold=21` مع مثال تدريب واحد (السؤال والمخرج المطلوب). إذا تكرر أي سؤال بين `train` و`valid` و`test`، فستظهر رسالة خطأ هنا.

In [ ]:
FAQ = {
    "deadline": "موعد تسليم الواجب هو 30 سبتمبر الساعة 23:59 (بتوقيت اليابان).",
    "destination": "قدّم الواجب عبر قسم «الواجبات» في بوابة التعلّم.",
    "format": "قدّم ملف PDF واحدًا باسم student_id_assignment_number.pdf.",
    "office_hours": "يمكنك طرح الأسئلة في ساعات الاستشارة كل يوم ثلاثاء من 17:00 إلى 18:00.",
    "late": "تُقبل الواجبات المتأخرة خلال 48 ساعة من الموعد النهائي، مع خصم من الدرجة.",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# مع دليل: (مفتاح FAQ، قوالب الأسئلة، الكلمات المطلوبة في الإجابة المتوقعة).
GROUNDED = {
    "deadline": (["متى موعد تسليم الواجب؟", "إلى متى يمكنني تسليم الواجب؟", "أخبرني بموعد التسليم", "ما الموعد النهائي للتسليم؟"], ["30 سبتمبر", "23:59"]),
    "destination": (["أين أقدّم الواجب؟", "أين أرفع الملف؟", "أين يمكنني تسليم الواجب؟", "ما جهة تسليم الواجب؟"], ["بوابة التعلّم", "الواجبات"]),
    "format": (["أخبرني بصيغة التسليم", "هل توجد قواعد لتسمية الملف؟", "بأي صيغة يجب أن أقدّم الملف؟", "هل يمكنني التسليم بصيغة غير PDF؟"], ["PDF"]),
    "office_hours": (["متى يمكنني طرح الأسئلة؟", "متى ساعات الاستشارة؟", "هل خُصص وقت لطرح الأسئلة على المدرّس؟", "ما الأوقات المتاحة للاستشارة؟"], ["ثلاثاء", "17:00"]),
    "late": (["ماذا يحدث إذا تأخرت في تسليم الواجب؟", "هل تُقبل الواجبات بعد الموعد النهائي؟", "ما قواعد التسليم المتأخر؟", "أخبرني كيف تُعامل الواجبات المقدّمة بعد الموعد النهائي"], ["48 ساعة", "خصم"]),
}
# بلا دليل: غير موجود في FAQ، لذا تكون needs_review=true صحيحة.
UNGROUNDED = [
    "هل يمكنني إعادة التسليم؟", "أخبرني كيف يُتعامل مع الغياب عن المحاضرات", "هل يمكنني الاطلاع على درجات المشاركين الآخرين؟",
    "متى تُعلَن الدرجات؟", "هل يمكن تمديد موعد التسليم؟", "هل يُسمح بالتسليم الجماعي؟",
    "كيف تُوزّع درجات الواجبات؟", "هل ستتاح تسجيلات المحاضرات؟",
]

# أسئلة إضافية للاختبار فقط (مستبعدة من train وvalid).
TEST_EXTRA_GROUNDED = {
    "deadline": ["أخبرني بتاريخ التسليم ووقته", "في أي تاريخ ووقت يحين موعد تسليم الواجب؟"],
    "destination": ["إلى أين أسلّم الواجب؟", "أخبرني أين أرسل واجبي"],
    "format": ["ما الصيغة المطلوبة للملف المقدّم؟", "ما صيغة الملفات وكم ملفًا عليّ تقديمه؟"],
    "office_hours": ["أخبرني بيوم ساعات الاستشارة ووقتها", "في أي يوم ومن أي ساعة يمكنني طرح الأسئلة؟"],
    "late": ["ماذا يحدث إذا فاتني موعد التسليم؟", "هل سيُقبل واجب يُقدَّم بعد الموعد النهائي؟"],
}
TEST_EXTRA_UNGROUNDED = [
    "هل يوجد حد لعدد الأحرف في الواجب؟", "هل سأتلقى ملاحظات على الواجب الذي قدمته؟",
    "هل توجد صيغة محددة للمراجع؟", "هل تتطلب ساعات الاستشارة حجز موعد؟",
]
UNGROUNDED_ANSWER = "لا تتضمن هذه الأسئلة المتكررة تلك المعلومة. يلزم التأكد من منظمي الدورة."

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # أول قالبين لـ train، والثالث لـ valid، والرابع لـ test (إبقاء الصياغات المختلفة منفصلة).
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # وزّع كل أربع أسئلة بلا دليل: اثنان لـ train، وواحد لـ valid، وواحد لـ test.
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["يلزم التأكد"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["يلزم التأكد"]}

# يضم train أربعة عشر مثالًا (10 بدليل و4 بلا دليل). اضبط التكرار بـ num_train_epochs في القسم 4 دون نسخ البيانات.
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "توجد أسئلة مكررة بين train وvalid وtest. راجع القوالب"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("مثال تدريب واحد | السؤال:", train[0]["question"], "| المخرج المطلوب:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. تحميل النموذج وتقييمه قبل التدريب

**ما ستفعله**: حمّل النموذج الأساسي دون تكميم (32 بتًا)، واجعله يجيب عن أسئلة `test` الواحد والعشرين، ثم قيّم النتائج.

يطلب الموجّه (`SYSTEM`) منه «إرجاع `answer` و`needs_review` دائمًا في سطر JSON واحد». أي إن **الصيغة مطلوبة منه بالفعل**. افحص هل يتبع النموذج هذا الطلب قبل التدريب.

**طريقة التقييم** (تُستخدم الدوال نفسها في القسم 5)

- **الصيغة**: يمكن تحليل المخرج بوصفه JSON، ويحتوي بالضبط على مفتاحَي `answer` و`needs_review`، بقيمتين من نوع سلسلة نصية وقيمة منطقية على الترتيب.
- **المحتوى**: تطابق قيمة `needs_review` القيمة المتوقعة، وتحتوي `answer` على الكلمات المطلوبة، مثل «30 سبتمبر» و«23:59».

يتطلب تقييم المحتوى تطابق قيمة `needs_review`؛ لذلك يحصل المخرج الذي ينقصه مفتاح على نتيجة «غير صحيح» في تقييم المحتوى حتى إن كانت صياغته صحيحة. افحص الصياغة بنفسك في الجدول.

في عمود سبب إخفاق الصيغة، يعني `json_error` تعذّر تحليل JSON، ويعني `not_object` أن الناتج ليس كائن JSON، ويعني `extra_keys` وجود مفاتيح إضافية، ويعني `missing_key` غياب مفتاح مطلوب، ويعني `wrong_type` أن نوع إحدى القيم غير صحيح. وتعرض أعمدة التقييم «صحيح» عند اجتياز الفحص و«غير صحيح» عند الإخفاق فيه.

يدعم `Qwen/Qwen3.5-0.8B` الصور أيضًا، لكن تحميله باستخدام `AutoModelForCausalLM` يستخدم الجزء النصي فقط.

**ما ستبحث عنه**

- الخلية الأولى: رسالة مثل `ذاكرة الأوزان (GB): 3.01`. يستغرق تنزيل النموذج (نحو 1.75GB) نحو دقيقة.
- الخلية الثانية: ملخص في سطر واحد (`format_rate` و`content_rate` وغيرهما) وجدول مخرجات الأسئلة الـ21. يستغرق التقييم نحو دقيقة. يذكر وصف التمرين الياباني الأصلي أن `needs_review` ينقص من مخرجات كثيرة قبل التدريب؛ افحص ما يحدث مع البيانات العربية.

**تحذيرات غير ضارة**

- `You are sending unauthenticated requests to the HF Hub`: يعني تنزيل النموذج دون تسجيل الدخول إلى Hugging Face. يمكن تنزيل النماذج العامة دون تسجيل الدخول. ويلزم تسجيل الدخول في الحالات الآتية، بغض النظر عن حجم النموذج:
  - استخدام نموذج يتطلب الموافقة على شروطه (نموذج مقيّد مثل Llama أو Gemma)، أو مستودع خاص.
  - رفع نماذجك أو بياناتك إلى Hub.
  - إرسال طلبات كثيرة خلال مدة قصيرة وبلوغ الحد. دون تسجيل الدخول، الحد المنشور هو 3,000 عملية جلب ملف لكل عنوان IP خلال خمس دقائق (الرقم المنشور في سبتمبر 2025؛ [حدود معدل Hub](https://huggingface.co/docs/hub/rate-limits)). يرفع تسجيل الدخول هذا الحد.

  النموذج في هذا الدفتر عام ولا يتطلب موافقة، ولا يتسبب تحميله إلا في عشرات قليلة من الطلبات، وهي أقل بكثير من الحد. إذا ظهر `429 Too Many Requests`، فستقرأ المكتبة مدة الانتظار وتعيد المحاولة تلقائيًا. إذا أصبح تسجيل الدخول ضروريًا، فسجّل رمز Hugging Face باسم `HF_TOKEN` عبر أيقونة المفتاح (Secrets) في الشريط الجانبي الأيسر من Colab، واسمح لهذا الدفتر بالوصول إليه؛ وسيُستخدم تلقائيًا.
- `... is falling back to its reference PyTorch implementation ...`: لم يُثبَّت تنفيذ سريع لبعض طبقات هذا النموذج (الانتباه الخطي)، لذا يُستخدم تنفيذ أبطأ لكنه صحيح. تظهر نحو أربعة أسطر.

**اقرأ قبل المتابعة**: راجع الجدول لترى «أي المفاتيح ناقصة» و«هل توافق الصياغة FAQ». في القسم 5، قارنه بمخرجات ما بعد التدريب. تُحفَظ المخرجات الأصلية أيضًا في `/content/qlora_lab/eval_before.json`؛ افتحها من قائمة الملفات على اليسار.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# دون تكميم. أبقِ الأوزان بدقة 32 بتًا (حدّد dtype لأن إغفاله يحمّل الدقة المنشورة).
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("تم التحميل:", CONFIG["base_model"], "| الجهاز:", DEVICE, "| ذاكرة الأوزان (GB):", BASE_MEMORY_GB)

# يحدد الموجّه الصيغة. افحص قبل التدريب هل يتبعها النموذج.
SYSTEM = "أجب عن الأسئلة المتعلقة بالأسئلة المتكررة للدورة. استند فقط إلى FAQ المقدّمة، وأجب دائمًا بسطر JSON واحد. الصيغة: {\"answer\": string, \"needs_review\": boolean}. للمعلومات غير الموجودة في FAQ، اكتب «يلزم التأكد» في answer واجعل needs_review تساوي true."

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\nالسؤال: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: اترك كتلة التفكير فارغة وأنتج الإجابة فقط (تحديد صريح لتوحيد الشروط بين النماذج).
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # اعرض المخرجات الأصلية دون اقتطاع.

def check_format(text: str):
    '''تحقق من تحليل JSON والمفاتيح المطلوبة والأنواع، وأعد سبب الإخفاق أيضًا.'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''ينجح المحتوى إذا طابقت needs_review القيمة المتوقعة وتضمنت answer الكلمات المطلوبة.'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "صحيح" if ok else "غير صحيح"

def evaluate(model, label: str):
    '''ولّد إجابات الأسئلة الـ21 وقيّمها، ثم أعد الملخص ونتيجة كل سؤال. احفظ المخرجات في eval_<label>.json.'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # إذا اختلت الصيغة لكن أمكن تحليل جزء شبيه بـ JSON، فقيّم المحتوى منفصلًا.
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# مخرجات كل سؤال. قارِن needs_review بالقيمة المتوقعة وافحص المفاتيح الناقصة واتفاق الصياغة مع FAQ.
display(pd.DataFrame([{"السؤال": r["question"], "قيمة needs_review المتوقعة": r["gold_needs_review"], "المخرج قبل التدريب": r["raw"],
                       "الصيغة": mark(r["format_ok"]), "سبب إخفاق الصيغة": r["format_reason"], "المحتوى": mark(r["content_ok"])} for r in before_rows]))

## 4. التدريب باستخدام LoRA ‏(استغرق 4–5 دقائق على T4 في التشغيل الياباني الأصلي)

### الفرق بين LoRA وQLoRA

- **LoRA**: تُثبَّت أوزان النموذج الأساسي، وتُدرَّب فقط مصفوفات صغيرة (المهايئات) تُضاف إلى بعض الطبقات.
- **QLoRA**: تُحفظ أوزان النموذج الأساسي مكمَّمة إلى 4 بتات، وتُدرَّب المهايئات فقط بالطريقة نفسها. المهايئات نفسها غير مكمَّمة.

يقلل QLoRA الذاكرة اللازمة لحفظ أوزان النموذج الأساسي. يستخدم نحو ربع ذاكرة تمثيل 16 بتًا، مما يتيح تدريب نماذج أكبر من سعة GPU المعتادة (لسبعة مليارات معلمة، نحو 14GB عند 16 بتًا مقابل نحو 3.5GB عند 4 بتات).

| الجانب | LoRA (دون تكميم) | QLoRA ‏(4 بتات) |
| --- | --- | --- |
| ذاكرة الأوزان | كبيرة | صغيرة (نحو ربع ذاكرة 16 بتًا) |
| مخرجات النموذج الأساسي | لا تتغير | قد تتغير بسبب خطأ التكميم؛ يعتمد حجم الأثر على النموذج |
| سرعة التدريب | مرجعية | ليست أسرع؛ وغالبًا أبطأ لأن أوزان 4 بتات تُحوَّل عند كل عملية حسابية |
| الاعتماديات | peft | peft مع bitsandbytes (تتطلب GPU من نوع CUDA) |

ملاحظات عند تطبيق QLoRA:

- مرّر `BitsAndBytesConfig(load_in_4bit=True, ...)` عند التحميل، ثم شغّل `prepare_model_for_kbit_training(model)` قبل إضافة LoRA.
- دقة التخزين (4 بتات) ودقة الحساب (`bnb_4bit_compute_dtype`، مثل 16 بتًا) إعدادان منفصلان.
- قارِن قبل التدريب وبعده مع ثبات شروط تكميم النموذج الأساسي. القياس قبل التدريب دون تكميم وبعده بدقة 4 بتات يخلط أثر التكميم بأثر التدريب.
- تُنشأ المهايئات للنموذج الأساسي المستخدم في التدريب. سجّل معًا معرّف النموذج الأساسي ومراجعته وإعدادات التكميم عند توزيعها أو إعادة استخدامها.
- لا يُنصح بتدريب بعض النماذج بدقة 4 بتات. يحذّر أحد [أدلة مكتبات التدريب](https://unsloth.ai/docs/models/qwen3.5/fine-tune) من تدريب Qwen3.5 بدقة 4 بتات لأن خطأ تكميمه أكبر من المعتاد.

يتسع نموذج هذا التمرين على T4 دون تكميم، لذا نتدرب باستخدام LoRA. ابدأ بـ LoRA إذا كان النموذج يتسع في الذاكرة؛ وإلا ففكّر في QLoRA أو نموذج أصغر. يمكنك فحص فروق الذاكرة والمخرجات عند التحميل بدقة 4 بتات في القسم 8 (للتوسّع).

### تشغيل التدريب

**ما ستفعله**: حوّل أمثلة `train` الأربعة عشر إلى صيغة محادثة (system / user / assistant)، وأضف LoRA، ودرب النموذج خلال 20 حقبة (40 خطوة). احفظ المهايئ المدرَّب في `adapter/`.

- تحدد `target_modules` الطبقات التي تُضاف إليها LoRA. نستهدف إسقاطات الانتباه (مثل `q_proj`) وإسقاطات MLP (مثل `gate_proj`). يستخدم هذا النموذج الانتباه الخطي في 18 طبقة من أصل 24؛ ولا تحتوي `q_proj` وما شابهها إلا الطبقات الست الباقية. وتحتوي كل الطبقات على MLP.
- حجم الدفعة 4 × تجميع التدرج 2 = 8 أمثلة لكل خطوة. مع 14 مثال تدريب، تتألف كل حقبة من خطوتين (8 أمثلة ثم 6)، فيصبح المجموع 40 خطوة خلال 20 حقبة. يُدرَّب كل مثال 20 مرة بالضبط. يُطبَّق إحماء معدل التعلم والاضمحلال وفق منحنى جيب التمام على هذه الخطوات الأربعين.
- يُحفَظ النموذج المدرَّب في `tuned`. ولأن LoRA تُضاف داخل طبقات النموذج الأساسي، فإن التوليد باستخدام المتغير الأصلي `model` يشمل المهايئات أيضًا. عطّل المهايئات مؤقتًا كما في القسم 6 عندما تحتاج إلى مخرجات ما قبل التدريب.

**ما ستبحث عنه**

- `trainable params: ... || all params: ... || trainable%: ...`: من خصائص LoRA تدريب نسبة صغيرة فقط من المعلمات كلها.
- جدول `Training Loss` (كل 10 خطوات): بدأت الخسارة قرب 1.5 وانخفضت إلى 0.01–0.02 عند الخطوة 40 في التشغيل الياباني الأصلي. قد تختلف القيم مع البيانات العربية؛ سجّل منحنى خسارتك، وإذا لم تنخفض فراجع إعدادات التدريب.
- `وقت التدريب: ... ثانية | الخطوات: 40 | الحقب: 20.0`: استغرق 240–290 ثانية على T4 في التشغيل الياباني الأصلي؛ قد يختلف الزمن مع البيانات العربية. يؤكد عدد الخطوات والحقب تنفيذ 14 مثال تدريب × 20 مرورًا كما حسبنا أعلاه.

**تحذير غير ضار**: تعني رسالة `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` أن إعدادات رمز النهاية وُفّقت مع محلّل الرموز النصية.

**حل المشكلات**: إذا توقف التدريب في منتصفه أو تعذّر تشغيله، فاقرأ القسم 4b أدناه.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # بيانات التدريب تربط المدخلات (system + user) بالمخرج المطلوب (assistant). استخدم build_messages نفسها كما في التقييم.
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# يستخدم Qwen3.5 الانتباه الخطي في 18 طبقة من أصل 24؛ ولا تتضمن q/k/v/o_proj إلا الطبقات الست الباقية. تتضمن جميع الطبقات gate/up/down_proj ‏(MLP).
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # اتركها عادةً دون تغيير. تتيح True حسابات مختلطة الدقة بـ 16 بتًا (CUDA فقط؛ تبقى الأوزان بدقة 32 بتًا). أسرع، لكن قد تصبح الخسارة nan.
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # نسبة المعلمات التي تُدرَّب باستخدام LoRA.
t0 = time.time()
trainer.train()  # راجع 4b لاستئناف التدريب إذا انقطع.
print(f"وقت التدريب: {time.time()-t0:.0f} ثانية | الخطوات: {trainer.state.global_step} | الحقب: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # يفعّل وضع التدريب dropout ويغيّر مخرجات التقييم (يمنع الإسناد عرض النموذج كاملًا).

### 4b. إذا توقف التدريب أو تعذّر تشغيله

**توقف في منتصفه (انقطع الاتصال ببيئة التشغيل مثلًا)**: تُحفظ نقاط التحقق كل 10 خطوات في `/content/qlora_lab/ckpt/`. إذا بقيت الملفات، فأعد تشغيل الأقسام 0–3، ثم غيّر `trainer.train()` في خلية التدريب أعلاه إلى `trainer.train(resume_from_checkpoint=True)` وشغّلها. يُستأنف التدريب من آخر نقطة تحقق حتى 40 خطوة (20 حقبة). إذا فُقدت بيئة التشغيل وملفاتها، فأعد التدريب من البداية أو استخدم البديل أدناه.

**تعذّر التدريب (لا تتوفر GPU أو لا يكفي الوقت)**: قارِن نتائج المدرّس وناقشها (المخرجات والدرجات بعد التدريب) من تشغيل الدفتر نفسه على T4. تحتاج خلايا القسم 5 وما بعده إلى النموذج المدرَّب `tuned`، لذا اقرأ الشرح إلى جانب نتائج المدرّس. عند التسليم، اضبط `executed_training_myself` في القسم 7 على `False`، واذكر بوضوح ما أنجزته بنفسك من الدفتر.

## 5. التقييم بعد التدريب (أسئلة `test` وإعدادات التوليد نفسها)

**ما ستفعله**: قيّم النموذج المدرَّب `tuned` على الأسئلة الـ21 نفسها وبإعدادات التوليد نفسها، وقارن النتائج قبل التدريب وبعده. التغيير الوحيد هو وجود المهايئات.

**ما ستبحث عنه**

- جدول المقارنة: كيف تغيّر `format_rate` (معدل الالتزام بالصيغة) و`content_rate` (دقة المحتوى) بعد التدريب؟
- `حالات التراجع`: الأسئلة التي نجحت في الصيغة والمحتوى قبل التدريب ثم فشلت بعده. من المرغوب أن تكون القائمة فارغة؛ قد تختلف النتائج مع البيانات العربية، لذا افحص قائمة تشغيلك.
- جدول كل سؤال: اقرأ الإخفاقات الباقية وحدد نوعها.
  - تطبيق بند FAQ مختلف وتقديم إجابة جازمة لسؤال بلا دليل (قيمة `needs_review` المتوقعة هي `True`).
  - التهرب من سؤال له دليل (القيمة المتوقعة `False`) بطلب التأكيد وقول «لم يُذكر».

يصف التمرين الياباني الأصلي ميل الإجابات بعد التدريب إلى إعادة نص التدريب (صياغة FAQ الأصلية) شبه حرفيًا واتساق الصيغة والعبارة، لكن ذلك لا يعني بالضرورة أن النموذج تعلّم الحكم على «وجود دليل في FAQ». افحص هل يظهر هذا السلوك في تشغيلك بالعربية.

لا يمكن تكرار تدريب GPU بصورة مطابقة تمامًا حتى مع تثبيت البذرة. قد يتقارب معدل الالتزام بالصيغة ودقة المحتوى بينما تختلف الأسئلة التي بقيت خاطئة بين تشغيل وآخر. اختلاف «الأسئلة الباقية» لديك عن زميلك طبيعي؛ قارن أنواع الإخفاقات.

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# أسئلة نجحت في الصيغة والمحتوى قبل التدريب ثم فشلت بعده (حالات التراجع).
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("حالات التراجع:", regressed)

# مخرجات كل سؤال الأصلية. افحص الصيغة بصريًا (JSON والمفاتيح والأنواع المطلوبة) والمحتوى (تطابق needs_review والكلمات المطلوبة).
detail = pd.DataFrame([{"السؤال": b["question"], "قيمة needs_review المتوقعة": b["gold_needs_review"],
                        "المخرج قبل التدريب": b["raw"], "الصيغة/المحتوى قبل التدريب": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "المخرج بعد التدريب": a["raw"], "الصيغة/المحتوى بعد التدريب": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. أسئلة المعرفة (تعلّم الصيغة لا يضمن المعرفة)

**ما ستفعله**: بعيدًا عن تجربة الصيغة، اطرح ثلاثة أسئلة واقعية لا توجد إجاباتها في FAQ ولا في بيانات التدريب (مع سؤال واحد تغطيه FAQ للمقارنة). الاستجابة المطلوبة هي «يلزم التأكد (`needs_review: true`)».

صنّف الاستجابات في ثلاث فئات.

| التصنيف | المعنى |
| --- | --- |
| `format_error` | لا يتبع صيغة JSON (شائع قبل التدريب) |
| `needs_review` | يقول «المعلومة غير مذكورة. يلزم التأكد». (وهي الاستجابة المطلوبة) |
| `asserted` | يقدّم جوابًا جازمًا مع `needs_review: false`. إذا غاب السؤال عن FAQ، فهذا ادعاء بلا دليل (اختلاق) |

**ما ستبحث عنه**: يصف التمرين الياباني الأصلي كثرة أخطاء الصيغة قبل التدريب واحتمال ظهور إجابات تطلب التأكد وأخرى جازمة بعده. قد تختلف النتائج مع البيانات العربية، فصنّف مخرجات تشغيلك الفعلية. يعرض الجدول المخرجات الأصلية أيضًا. في الإجابات الجازمة، اقرأ ما يبدو أن النموذج استند إليه (هل أعاد استخدام بند FAQ آخر؟). اذكر نتائج مجموعة البيانات هذه بدل أن تفترض سلفًا أن «الضبط الدقيق لا يستطيع تصحيح المعرفة مطلقًا».

تُنتج مخرجات ما قبل التدريب بتعطيل المهايئات مؤقتًا باستخدام `tuned.disable_adapter()`.

In [ ]:
KNOWLEDGE_QUESTIONS = [
    "متى موعد تسليم الواجب النهائي لهذه الدورة؟",       # غير موجود في FAQ (افتراضي).
    "ما اسم المدرّس؟",                     # غير موجود في FAQ.
    "ما موضوع المحاضرة القادمة؟",                 # غير موجود في FAQ.
    "متى موعد تسليم الواجب؟",                   # موجود في FAQ (للمقارنة).
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # أُضيفت LoRA إلى طبقات النموذج الأساسي، لذا يستخدم التوليد بـ model المهايئات بعد التدريب أيضًا. عطّلها مؤقتًا للحصول على مخرجات ما قبل التدريب.
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"السؤال": q, "موجود في FAQ": q in grounded_questions, "التصنيف قبل التدريب": classify(before_raw), "المخرج قبل التدريب": before_raw,
                 "التصنيف بعد التدريب": classify(after_raw), "المخرج بعد التدريب": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. جدول المقارنة والتسليم

**ما ستفعله**: اجمع الإعدادات وإصدارات المكتبات وملخصَي ما قبل التدريب وبعده والأسئلة التي تراجع أداؤها في ملف `result.json` واحد.

**ما ستقدّمه**

- `result.json` و`eval_before.json` و`eval_after.json` (في `/content/qlora_lab`؛ انقر بزر الفأرة الأيمن على الملف في القائمة يسارًا ← **Download**. إذا كانت `USE_DRIVE = True`، تُنسخ الملفات إلى Drive أيضًا).
- جدول المقارنة من القسم 5 وشرح الإخفاقات الباقية (أنواع الإخفاق).
- نتائج القسم 6 (الإجابات التي أصبحت تطلب التأكد، وتلك التي بقيت جازمة).
- مدى تقدمك الشخصي في تشغيل التدريب. إذا استخدمت نتائج المدرّس، فاضبط `executed_training_myself` على `False`.

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (دون تكميم)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # اضبطها على False إذا استخدمت نتائج المدرّس.
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("حُفظ في Drive:", CONFIG["drive_dir"])

## 8. للتوسّع: التحميل بتكميم 4 بتات ومقارنة الذاكرة والمخرجات (اختياري)

**ما ستفعله**: حمّل النموذج نفسه بدقة 4 بتات (NF4)، وافحص تغيّر ذاكرة الأوزان ومخرجات ما قبل التدريب. تضيف QLoRA طريقة LoRA إلى هذا النموذج الأساسي ذي 4 بتات وتدرّبه. لن نجري تدريبًا هنا.

- تتطلب bitsandbytes وحدة GPU من نوع CUDA، لذا لا تعمل هذه الخلية إلا في بيئة تشغيل مع GPU. يستغرق التثبيت نحو دقيقة.
- يضع التحميل بدقة 4 بتات النموذج على GPU باستخدام `device_map="auto"` (وفق طريقة تحميل bitsandbytes).
- إذا جرّبت تدريب النموذج ذي 4 بتات أيضًا، فشغّل `prepare_model_for_kbit_training(model_4bit)` أولًا، واتبع خطوات التدريب في القسم 4، وقيّم ما قبل التدريب وبعده باستخدام نموذج أساسي بدقة 4 بتات في الحالتين.

**ما ستبحث عنه**

- ذاكرة الأوزان: تقل عن 1GB عند 4 بتات، مقابل نحو 3GB عند 32 بتًا.
- جدول المقارنة وجدول كل سؤال: لأن التدريب لم يحدث، فإن الفروق عن نتائج ما قبل التدريب بدقة 32 بتًا (`before`) تعكس أثر التكميم. يصف التمرين الياباني الأصلي تحسّن بعض الأسئلة وظهور قيمة `needs_review` في مخرجات أخرى كسلسلة نصية `"false"` تفشل في فحص النوع، أو إجابة النموذج «نعم» أو «لا شيء» عن أسئلة بلا دليل. افحص مخرجاتك بالعربية لتعرف أي تغييرات حدثت فعلًا واتجاهها.

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # دقة التخزين: 4 بتات (NF4).
                         bnb_4bit_compute_dtype=torch.float16,             # دقة الحساب: 16 بتًا (لا يدعم T4 الصيغة bf16).
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("ذاكرة الأوزان (GB) | دون تكميم (32 بتًا):", BASE_MEMORY_GB, "| 4 بتات:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# لم يحدث تدريب، لذا تعكس الفروق عن نتائج ما قبل التدريب دون تكميم (before) أثر التكميم.
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"السؤال": r["question"], "قيمة needs_review المتوقعة": r["gold_needs_review"], "المخرج قبل التدريب بدقة 4 بتات": r["raw"],
                       "الصيغة/المحتوى": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "سبب إخفاق الصيغة": r["format_reason"]} for r in q_rows]))